In [2]:
import zipfile
import os

zip_path = "../data/SWPK.zip"

with zipfile.ZipFile(zip_path, "r") as zip_file:
    print("Files inside the ZIP:")
    for file in zip_file.namelist():
        print(file)

Files inside the ZIP:
datadump_s5-000.csv
datadump_s5-001.csv
datadump_s5-002.csv
datadump_s5-003.csv
datadump_s5-004.csv
datadump_s5-005.csv
datadump_s5-006.csv
datadump_s5-007.csv
datadump_s5-008.csv
datadump_s5-009.csv
datadump_s5-010.csv
datadump_s5-011.csv
datadump_s5-012.csv
datadump_s5-013.csv
datadump_s5-014.csv
datadump_s5-015.csv
datadump_s5-016.csv
datadump_s5-017.csv
datadump_s5-018.csv
datadump_s5-019.csv
datadump_s5-020.csv
datadump_s5-021.csv


In [3]:
import zipfile
import pandas as pd

zip_path = "../data/SWPK.zip"

with zipfile.ZipFile(zip_path, "r") as zip_file:
    with zip_file.open("datadump_s5-000.csv") as csv_file:
        df = pd.read_csv(csv_file, nrows=1000)

print("Rows:", len(df))
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

Rows: 1000

Columns:
['dateid', 'platform', 'gamemode', 'mapname', 'matchid', 'roundnumber', 'objectivelocation', 'winrole', 'endroundreason', 'roundduration', 'clearancelevel', 'skillrank', 'role', 'team', 'haswon', 'operator', 'nbkills', 'isdead', 'primaryweapon', 'primaryweapontype', 'primarysight', 'primarygrip', 'primaryunderbarrel', 'primarybarrel', 'secondaryweapon', 'secondaryweapontype', 'secondarysight', 'secondarygrip', 'secondaryunderbarrel', 'secondarybarrel', 'secondarygadget']

First 5 rows:


,dateid,platform,gamemode,mapname,matchid,roundnumber,objectivelocation,winrole,endroundreason,roundduration,...,primarygrip,primaryunderbarrel,primarybarrel,secondaryweapon,secondaryweapontype,secondarysight,secondarygrip,secondaryunderbarrel,secondarybarrel,secondarygadget
0,20170212,PC,HOSTAGE,CLUB_HOUSE,1522380841,1,STRIP_CLUB,Defender,AttackersKilledHostage,124,...,Vertical,NaN,Compensator,5.7_USG,Pistols,NaN,NaN,NaN,NaN,IMPACT_GRENADE
1,20170212,PC,HOSTAGE,CLUB_HOUSE,1522380841,4,CHURCH,Defender,AttackersEliminated,217,...,Vertical,Laser,Suppressor,P12,Pistols,NaN,NaN,Laser,Suppressor,DEPLOYABLE_SHIELD
2,20170212,PC,HOSTAGE,CLUB_HOUSE,1522380841,3,CHURCH,Defender,AttackersEliminated,160,...,NaN,NaN,NaN,MK1_9mm,Pistols,NaN,NaN,NaN,NaN,DEPLOYABLE_SHIELD
3,20170212,PC,HOSTAGE,CLUB_HOUSE,1522380841,4,CHURCH,Defender,AttackersEliminated,217,...,NaN,NaN,MuzzleBrake,PRB92,Pistols,NaN,NaN,NaN,NaN,IMPACT_GRENADE
4,20170212,PC,HOSTAGE,CLUB_HOUSE,1522380841,6,BEDROOM,Attacker,DefendersEliminated,143,...,Vertical,Laser,Suppressor,P12,Pistols,NaN,NaN,Laser,Suppressor,DEPLOYABLE_SHIELD


In [4]:
# Look at one specific match and round
sample = df[
    (df["matchid"] == 1522380841) &
    (df["roundnumber"] == 4)
]

print(sample[["matchid", "roundnumber", "role", "team", "operator", "winrole", "haswon"]])
print("\nNumber of players:", len(sample))

       matchid  roundnumber      role  team             operator   winrole  \
1   1522380841            4  Defender     0           GSG9-JAGER  Defender   
3   1522380841            4  Defender     0         BOPE-CAVEIRA  Defender   
15  1522380841            4  Attacker     1              GSG9-IQ  Defender   
17  1522380841            4  Attacker     1  NAVYSEAL-BLACKBEARD  Defender   
20  1522380841            4  Defender     0    SPETSNAZ-TACHANKA  Defender   
21  1522380841            4  Defender     0          GSG9-BANDIT  Defender   
27  1522380841            4  Attacker     1           SAT-HIBANA  Defender   
34  1522380841            4  Attacker     1        G.E.O.-JACKAL  Defender   
51  1522380841            4  Attacker     1             SWAT-ASH  Defender   

    haswon  
1        1  
3        1  
15       0  
17       0  
20       1  
21       1  
27       0  
34       0  
51       0  

Number of players: 9


In [5]:
# Count how many attackers and defenders each round has

round_counts = (
    df.groupby(["matchid", "roundnumber", "role"])
      .size()
      .unstack(fill_value=0)
)

print(round_counts.head(20))

role                    Attacker  Defender
matchid    roundnumber                    
1522380841 1                   5         5
           2                   5         4
           3                   3         5
           4                   5         4
           5                   4         5
           6                   5         4
1522488121 1                   5         5
           2                   5         5
           3                   5         5
           4                   5         5
1522514641 1                   5         5
           2                   4         5
           3                   5         4
           4                   4         5
1522741281 1                   4         5
           2                   5         4
           3                   4         5
           4                   5         4
           5                   4         5
1522760761 1                   5         5


In [6]:
# Look at the relationship between winrole and haswon
print(
    df[["role", "winrole", "haswon"]]
    .drop_duplicates()
    .sort_values(["role", "winrole", "haswon"])
)

       role   winrole  haswon
6  Attacker  Attacker       1
8  Attacker  Defender       0
4  Defender  Attacker       0
0  Defender  Defender       1


In [7]:
# Check whether each round has one consistent winner
winner_check = (
    df.groupby(["matchid", "roundnumber"])["winrole"]
      .nunique()
)

print("Rounds with exactly one winner:", (winner_check == 1).sum())
print("Rounds with multiple winners:", (winner_check > 1).sum())

Rounds with exactly one winner: 108
Rounds with multiple winners: 0


In [8]:
# Count the number of attackers and defenders in each round
round_counts = (
    df.groupby(["matchid", "roundnumber", "role"])
      .size()
      .unstack(fill_value=0)
)

# Keep only complete 5v5 rounds
complete_rounds = round_counts[
    (round_counts["Attacker"] == 5) &
    (round_counts["Defender"] == 5)
]

print("Complete 5v5 rounds:", len(complete_rounds))

Complete 5v5 rounds: 65


In [9]:
# Get the IDs of all complete 5v5 rounds
complete_round_ids = complete_rounds.index

# Keep only rows belonging to those rounds
clean_df = df.set_index(["matchid", "roundnumber"]).loc[
    complete_round_ids
].reset_index()

print("Rows in cleaned dataset:", len(clean_df))
print(clean_df.head())

Rows in cleaned dataset: 650
      matchid  roundnumber    dateid platform gamemode     mapname  \
0  1522380841            1  20170212       PC  HOSTAGE  CLUB_HOUSE   
1  1522380841            1  20170212       PC  HOSTAGE  CLUB_HOUSE   
2  1522380841            1  20170212       PC  HOSTAGE  CLUB_HOUSE   
3  1522380841            1  20170212       PC  HOSTAGE  CLUB_HOUSE   
4  1522380841            1  20170212       PC  HOSTAGE  CLUB_HOUSE   

  objectivelocation   winrole          endroundreason  roundduration  ...  \
0        STRIP_CLUB  Defender  AttackersKilledHostage            124  ...   
1        STRIP_CLUB  Defender  AttackersKilledHostage            124  ...   
2        STRIP_CLUB  Defender  AttackersKilledHostage            124  ...   
3        STRIP_CLUB  Defender  AttackersKilledHostage            124  ...   
4        STRIP_CLUB  Defender  AttackersKilledHostage            124  ...   

   primarygrip primaryunderbarrel primarybarrel  secondaryweapon  \
0     Vertical     

In [10]:
# Pick one complete round
example_round = clean_df[
    (clean_df["matchid"] == 1522488121) &
    (clean_df["roundnumber"] == 1)
]

# Display only the columns we care about right now
print(
    example_round[
        [
            "matchid",
            "roundnumber",
            "mapname",
            "objectivelocation",
            "role",
            "operator",
            "winrole"
        ]
    ].to_string(index=False)
)

   matchid  roundnumber mapname             objectivelocation     role          operator  winrole
1522488121            1   PLANE MEETING_ROOM-EXECUTIVE_OFFICE Attacker     SWAT-THERMITE Defender
1522488121            1   PLANE MEETING_ROOM-EXECUTIVE_OFFICE Defender        SWAT-PULSE Defender
1522488121            1   PLANE MEETING_ROOM-EXECUTIVE_OFFICE Attacker     G.E.O.-JACKAL Defender
1522488121            1   PLANE MEETING_ROOM-EXECUTIVE_OFFICE Attacker     SPETSNAZ-GLAZ Defender
1522488121            1   PLANE MEETING_ROOM-EXECUTIVE_OFFICE Attacker        SAS-SLEDGE Defender
1522488121            1   PLANE MEETING_ROOM-EXECUTIVE_OFFICE Attacker     GIGN-MONTAGNE Defender
1522488121            1   PLANE MEETING_ROOM-EXECUTIVE_OFFICE Defender        JTF2-FROST Defender
1522488121            1   PLANE MEETING_ROOM-EXECUTIVE_OFFICE Defender        GSG9-JAGER Defender
1522488121            1   PLANE MEETING_ROOM-EXECUTIVE_OFFICE Defender NAVYSEAL-VALKYRIE Defender
1522488121          

In [11]:
# Get the attackers
attackers = example_round[
    example_round["role"] == "Attacker"
]["operator"].tolist()

# Get the defenders
defenders = example_round[
    example_round["role"] == "Defender"
]["operator"].tolist()

# Get the winner
winner = example_round["winrole"].iloc[0]

print("Attackers:")
print(attackers)

print("\nDefenders:")
print(defenders)

print("\nWinner:")
print(winner)

Attackers:
['SWAT-THERMITE', 'G.E.O.-JACKAL', 'SPETSNAZ-GLAZ', 'SAS-SLEDGE', 'GIGN-MONTAGNE']

Defenders:
['SWAT-PULSE', 'JTF2-FROST', 'GSG9-JAGER', 'NAVYSEAL-VALKYRIE', 'G.E.O.-MIRA']

Winner:
Defender


In [12]:
# Create one row for each complete round
round_data = []

# Go through each unique match + round combination
for (matchid, roundnumber), group in clean_df.groupby(
    ["matchid", "roundnumber"]
):
    
    # Separate attackers and defenders
    attackers = group[group["role"] == "Attacker"]["operator"].tolist()
    defenders = group[group["role"] == "Defender"]["operator"].tolist()

    # We only want complete 5v5 rounds
    if len(attackers) != 5 or len(defenders) != 5:
        continue

    # Get information that is the same for every player in the round
    mapname = group["mapname"].iloc[0]
    objectivelocation = group["objectivelocation"].iloc[0]
    winrole = group["winrole"].iloc[0]

    # Convert winner into 1/0
    attack_win = 1 if winrole == "Attacker" else 0

    # Store the round
    round_data.append({
        "matchid": matchid,
        "roundnumber": roundnumber,
        "mapname": mapname,
        "objectivelocation": objectivelocation,
        "attackers": attackers,
        "defenders": defenders,
        "attack_win": attack_win
    })

# Convert our list into a DataFrame
round_df = pd.DataFrame(round_data)

print("Number of complete rounds:", len(round_df))
display(round_df.head())

Number of complete rounds: 65


,matchid,roundnumber,mapname,objectivelocation,attackers,defenders,attack_win
0,1522380841,1,CLUB_HOUSE,STRIP_CLUB,"[GIGN-TWITCH, SPETSNAZ-FUZE, SWAT-ASH, SPETSNA...","[SWAT-CASTLE, JTF2-FROST, G.E.O.-MIRA, NAVYSEA...",0
1,1522488121,1,PLANE,MEETING_ROOM-EXECUTIVE_OFFICE,"[SWAT-THERMITE, G.E.O.-JACKAL, SPETSNAZ-GLAZ, ...","[SWAT-PULSE, JTF2-FROST, GSG9-JAGER, NAVYSEAL-...",0
2,1522488121,2,PLANE,CARGO_HOLD-LUGGAGE_HOLD,"[G.E.O.-JACKAL, SPETSNAZ-FUZE, JTF2-BUCK, SWAT...","[SWAT-PULSE, GSG9-BANDIT, NAVYSEAL-VALKYRIE, G...",1
3,1522488121,3,PLANE,STAFF_SECTION-EXECUTIVE_BEDROOM,"[SPETSNAZ-GLAZ, GIGN-TWITCH, SWAT-ASH, SWAT-TH...","[G.E.O.-MIRA, NAVYSEAL-VALKYRIE, SWAT-PULSE, G...",0
4,1522488121,4,PLANE,CARGO_HOLD-LUGGAGE_HOLD,"[SPETSNAZ-GLAZ, SWAT-ASH, SWAT-RESERVE, GSG9-I...","[GIGN-DOC, SAT-ECHO, GSG9-JAGER, GSG9-BANDIT, ...",1


In [13]:
# Remove the faction prefix from an operator name
def clean_operator_name(operator):
    return operator.split("-")[-1]


# Clean the attackers
round_df["attackers"] = round_df["attackers"].apply(
    lambda operators: [
        clean_operator_name(operator)
        for operator in operators
    ]
)

# Clean the defenders
round_df["defenders"] = round_df["defenders"].apply(
    lambda operators: [
        clean_operator_name(operator)
        for operator in operators
    ]
)

display(round_df.head())

,matchid,roundnumber,mapname,objectivelocation,attackers,defenders,attack_win
0,1522380841,1,CLUB_HOUSE,STRIP_CLUB,"[TWITCH, FUZE, ASH, GLAZ, HIBANA]","[CASTLE, FROST, MIRA, VALKYRIE, MUTE]",0
1,1522488121,1,PLANE,MEETING_ROOM-EXECUTIVE_OFFICE,"[THERMITE, JACKAL, GLAZ, SLEDGE, MONTAGNE]","[PULSE, FROST, JAGER, VALKYRIE, MIRA]",0
2,1522488121,2,PLANE,CARGO_HOLD-LUGGAGE_HOLD,"[JACKAL, FUZE, BUCK, ASH, GLAZ]","[PULSE, BANDIT, VALKYRIE, JAGER, MIRA]",1
3,1522488121,3,PLANE,STAFF_SECTION-EXECUTIVE_BEDROOM,"[GLAZ, TWITCH, ASH, THERMITE, IQ]","[MIRA, VALKYRIE, PULSE, JAGER, BANDIT]",0
4,1522488121,4,PLANE,CARGO_HOLD-LUGGAGE_HOLD,"[GLAZ, ASH, RESERVE, IQ, JACKAL]","[DOC, ECHO, JAGER, BANDIT, MIRA]",1


In [14]:
# Get every unique attacker and defender operator
all_attackers = set(
    operator
    for operators in round_df["attackers"]
    for operator in operators
)

all_defenders = set(
    operator
    for operators in round_df["defenders"]
    for operator in operators
)

# Combine them
all_operators = sorted(all_attackers | all_defenders)

print("Number of unique operators:", len(all_operators))
print("\nOperators:")
print(all_operators)

print("\nNumber of maps:", round_df["mapname"].nunique())
print("Maps:")
print(round_df["mapname"].unique())

print("\nNumber of sites:", round_df["objectivelocation"].nunique())

Number of unique operators: 31

Operators:
['ASH', 'BANDIT', 'BLACKBEARD', 'BLITZ', 'BUCK', 'CAPITAO', 'CASTLE', 'CAVEIRA', 'DOC', 'ECHO', 'FROST', 'FUZE', 'GLAZ', 'HIBANA', 'IQ', 'JACKAL', 'JAGER', 'KAPKAN', 'MIRA', 'MONTAGNE', 'MUTE', 'PULSE', 'RESERVE', 'ROOK', 'SLEDGE', 'SMOKE', 'TACHANKA', 'THATCHER', 'THERMITE', 'TWITCH', 'VALKYRIE']

Number of maps: 9
Maps:
<StringArray>
['CLUB_HOUSE',      'PLANE',      'KANAL',  'CONSULATE',      'YACHT',
     'OREGON',     'BORDER', 'SKYSCRAPER',       'BANK']
Length: 9, dtype: str

Number of sites: 29


In [16]:
# Recreate round_df from clean_df

round_data = []

for (matchid, roundnumber), group in clean_df.groupby(
    ["matchid", "roundnumber"]
):

    attackers = group[
        group["role"] == "Attacker"
    ]["operator"].tolist()

    defenders = group[
        group["role"] == "Defender"
    ]["operator"].tolist()

    if len(attackers) != 5 or len(defenders) != 5:
        continue

    mapname = group["mapname"].iloc[0]
    objectivelocation = group["objectivelocation"].iloc[0]
    winrole = group["winrole"].iloc[0]

    attack_win = 1 if winrole == "Attacker" else 0

    round_data.append({
        "matchid": matchid,
        "roundnumber": roundnumber,
        "mapname": mapname,
        "objectivelocation": objectivelocation,
        "attackers": attackers,
        "defenders": defenders,
        "attack_win": attack_win
    })

round_df = pd.DataFrame(round_data)

# Clean the faction prefixes again
def clean_operator_name(operator):
    return operator.split("-")[-1]

round_df["attackers"] = round_df["attackers"].apply(
    lambda operators: [
        clean_operator_name(operator)
        for operator in operators
    ]
)

round_df["defenders"] = round_df["defenders"].apply(
    lambda operators: [
        clean_operator_name(operator)
        for operator in operators
    ]
)

print("Rounds:", len(round_df))

Rounds: 65


In [17]:
reserve_rounds = round_df[
    round_df["attackers"].apply(lambda x: "RESERVE" in x) |
    round_df["defenders"].apply(lambda x: "RESERVE" in x)
]

print("Rounds containing RESERVE:", len(reserve_rounds))

display(
    reserve_rounds[
        [
            "matchid",
            "roundnumber",
            "mapname",
            "objectivelocation",
            "attackers",
            "defenders",
            "attack_win"
        ]
    ]
)

Rounds containing RESERVE: 3


,matchid,roundnumber,mapname,objectivelocation,attackers,defenders,attack_win
4,1522488121,4,PLANE,CARGO_HOLD-LUGGAGE_HOLD,"[GLAZ, ASH, RESERVE, IQ, JACKAL]","[DOC, ECHO, JAGER, BANDIT, MIRA]",1
28,1523847781,7,PLANE,MEETING_ROOM,"[HIBANA, JACKAL, TWITCH, SLEDGE, GLAZ]","[BANDIT, RESERVE, JAGER, FROST, MIRA]",1
40,1523909061,5,BORDER,OFFICES,"[THERMITE, RESERVE, RESERVE, JACKAL, RESERVE]","[CAVEIRA, CASTLE, BANDIT, PULSE, JAGER]",0


In [18]:
# Remove rounds that contain RESERVE
round_df = round_df[
    ~round_df["attackers"].apply(lambda x: "RESERVE" in x) &
    ~round_df["defenders"].apply(lambda x: "RESERVE" in x)
].reset_index(drop=True)

print("Rounds remaining:", len(round_df))

Rounds remaining: 62


In [19]:
print(
    "RESERVE remaining:",
    sum(round_df["attackers"].apply(lambda x: "RESERVE" in x)) +
    sum(round_df["defenders"].apply(lambda x: "RESERVE" in x))
)

RESERVE remaining: 0


In [20]:
print(round_df["attack_win"].value_counts())

attack_win
1    32
0    30
Name: count, dtype: int64


In [21]:
operator_roles = (
    clean_df[["role", "operator"]]
    .drop_duplicates()
    .sort_values(["role", "operator"])
)

display(operator_roles)

,role,operator
99,Attacker,BOPE-CAPITAO
12,Attacker,G.E.O.-JACKAL
15,Attacker,GIGN-MONTAGNE
1,Attacker,GIGN-TWITCH
55,Attacker,GSG9-BLITZ
39,Attacker,GSG9-IQ
403,Attacker,GSG9-RESERVE
23,Attacker,JTF2-BUCK
59,Attacker,NAVYSEAL-BLACKBEARD
14,Attacker,SAS-SLEDGE


In [22]:
# Recreate the round-level dataset from the original clean_df

round_data = []

for (matchid, roundnumber), group in clean_df.groupby(
    ["matchid", "roundnumber"]
):
    
    attackers = group[
        group["role"] == "Attacker"
    ]["operator"].tolist()

    defenders = group[
        group["role"] == "Defender"
    ]["operator"].tolist()

    # Only keep complete 5v5 rounds
    if len(attackers) != 5 or len(defenders) != 5:
        continue

    mapname = group["mapname"].iloc[0]
    objectivelocation = group["objectivelocation"].iloc[0]
    winrole = group["winrole"].iloc[0]

    attack_win = 1 if winrole == "Attacker" else 0

    round_data.append({
        "matchid": matchid,
        "roundnumber": roundnumber,
        "mapname": mapname,
        "objectivelocation": objectivelocation,
        "attackers": attackers,
        "defenders": defenders,
        "attack_win": attack_win
    })

round_df = pd.DataFrame(round_data)


# Remove the faction prefix
def clean_operator_name(operator):
    return operator.split("-")[-1]


round_df["attackers"] = round_df["attackers"].apply(
    lambda operators: [
        clean_operator_name(operator)
        for operator in operators
    ]
)

round_df["defenders"] = round_df["defenders"].apply(
    lambda operators: [
        clean_operator_name(operator)
        for operator in operators
    ]
)


# Change RESERVE to RECRUIT
round_df["attackers"] = round_df["attackers"].apply(
    lambda operators: [
        "RECRUIT" if operator == "RESERVE" else operator
        for operator in operators
    ]
)

round_df["defenders"] = round_df["defenders"].apply(
    lambda operators: [
        "RECRUIT" if operator == "RESERVE" else operator
        for operator in operators
    ]
)


print("Complete rounds:", len(round_df))
display(round_df.head())

Complete rounds: 65


,matchid,roundnumber,mapname,objectivelocation,attackers,defenders,attack_win
0,1522380841,1,CLUB_HOUSE,STRIP_CLUB,"[TWITCH, FUZE, ASH, GLAZ, HIBANA]","[CASTLE, FROST, MIRA, VALKYRIE, MUTE]",0
1,1522488121,1,PLANE,MEETING_ROOM-EXECUTIVE_OFFICE,"[THERMITE, JACKAL, GLAZ, SLEDGE, MONTAGNE]","[PULSE, FROST, JAGER, VALKYRIE, MIRA]",0
2,1522488121,2,PLANE,CARGO_HOLD-LUGGAGE_HOLD,"[JACKAL, FUZE, BUCK, ASH, GLAZ]","[PULSE, BANDIT, VALKYRIE, JAGER, MIRA]",1
3,1522488121,3,PLANE,STAFF_SECTION-EXECUTIVE_BEDROOM,"[GLAZ, TWITCH, ASH, THERMITE, IQ]","[MIRA, VALKYRIE, PULSE, JAGER, BANDIT]",0
4,1522488121,4,PLANE,CARGO_HOLD-LUGGAGE_HOLD,"[GLAZ, ASH, RECRUIT, IQ, JACKAL]","[DOC, ECHO, JAGER, BANDIT, MIRA]",1


In [23]:
print(
    round_df[
        round_df["attackers"].apply(lambda x: "RECRUIT" in x) |
        round_df["defenders"].apply(lambda x: "RECRUIT" in x)
    ][["attackers", "defenders", "attack_win"]]
)

                                        attackers  \
4                [GLAZ, ASH, RECRUIT, IQ, JACKAL]   
28         [HIBANA, JACKAL, TWITCH, SLEDGE, GLAZ]   
40  [THERMITE, RECRUIT, RECRUIT, JACKAL, RECRUIT]   

                                  defenders  attack_win  
4          [DOC, ECHO, JAGER, BANDIT, MIRA]           1  
28    [BANDIT, RECRUIT, JAGER, FROST, MIRA]           1  
40  [CAVEIRA, CASTLE, BANDIT, PULSE, JAGER]           0  


In [24]:
# Get all unique attacker operators
attack_operators = sorted(
    set(
        operator
        for operators in round_df["attackers"]
        for operator in operators
    )
)

# Get all unique defender operators
defense_operators = sorted(
    set(
        operator
        for operators in round_df["defenders"]
        for operator in operators
    )
)

print("Attack operators:", len(attack_operators))
print(attack_operators)

print("\nDefense operators:", len(defense_operators))
print(defense_operators)

Attack operators: 16
['ASH', 'BLACKBEARD', 'BLITZ', 'BUCK', 'CAPITAO', 'FUZE', 'GLAZ', 'HIBANA', 'IQ', 'JACKAL', 'MONTAGNE', 'RECRUIT', 'SLEDGE', 'THATCHER', 'THERMITE', 'TWITCH']

Defense operators: 16
['BANDIT', 'CASTLE', 'CAVEIRA', 'DOC', 'ECHO', 'FROST', 'JAGER', 'KAPKAN', 'MIRA', 'MUTE', 'PULSE', 'RECRUIT', 'ROOK', 'SMOKE', 'TACHANKA', 'VALKYRIE']


In [25]:
# Create a copy so we don't accidentally modify round_df
features_df = round_df.copy()

# Create one column for each attacker operator
for operator in attack_operators:
    features_df[f"ATTACK_{operator}"] = (
        features_df["attackers"]
        .apply(lambda team: int(operator in team))
    )

# Create one column for each defender operator
for operator in defense_operators:
    features_df[f"DEFENSE_{operator}"] = (
        features_df["defenders"]
        .apply(lambda team: int(operator in team))
    )

In [26]:
print("Rows:", len(features_df))
print("Columns:", len(features_df.columns))
display(features_df.head())

Rows: 65
Columns: 39


,matchid,roundnumber,mapname,objectivelocation,attackers,defenders,attack_win,ATTACK_ASH,ATTACK_BLACKBEARD,ATTACK_BLITZ,...,DEFENSE_JAGER,DEFENSE_KAPKAN,DEFENSE_MIRA,DEFENSE_MUTE,DEFENSE_PULSE,DEFENSE_RECRUIT,DEFENSE_ROOK,DEFENSE_SMOKE,DEFENSE_TACHANKA,DEFENSE_VALKYRIE
0,1522380841,1,CLUB_HOUSE,STRIP_CLUB,"[TWITCH, FUZE, ASH, GLAZ, HIBANA]","[CASTLE, FROST, MIRA, VALKYRIE, MUTE]",0,1,0,0,...,0,0,1,1,0,0,0,0,0,1
1,1522488121,1,PLANE,MEETING_ROOM-EXECUTIVE_OFFICE,"[THERMITE, JACKAL, GLAZ, SLEDGE, MONTAGNE]","[PULSE, FROST, JAGER, VALKYRIE, MIRA]",0,0,0,0,...,1,0,1,0,1,0,0,0,0,1
2,1522488121,2,PLANE,CARGO_HOLD-LUGGAGE_HOLD,"[JACKAL, FUZE, BUCK, ASH, GLAZ]","[PULSE, BANDIT, VALKYRIE, JAGER, MIRA]",1,1,0,0,...,1,0,1,0,1,0,0,0,0,1
3,1522488121,3,PLANE,STAFF_SECTION-EXECUTIVE_BEDROOM,"[GLAZ, TWITCH, ASH, THERMITE, IQ]","[MIRA, VALKYRIE, PULSE, JAGER, BANDIT]",0,1,0,0,...,1,0,1,0,1,0,0,0,0,1
4,1522488121,4,PLANE,CARGO_HOLD-LUGGAGE_HOLD,"[GLAZ, ASH, RECRUIT, IQ, JACKAL]","[DOC, ECHO, JAGER, BANDIT, MIRA]",1,1,0,0,...,1,0,1,0,0,0,0,0,0,0


In [27]:
# Create one-hot features for maps
map_features = pd.get_dummies(
    features_df["mapname"],
    prefix="MAP",
    dtype=int
)

# Create one-hot features for objective sites
site_features = pd.get_dummies(
    features_df["objectivelocation"],
    prefix="SITE",
    dtype=int
)

# Add them to our feature DataFrame
features_df = pd.concat(
    [features_df, map_features, site_features],
    axis=1
)

print("Rows:", len(features_df))
print("Columns:", len(features_df.columns))

Rows: 65
Columns: 77


In [28]:
display(features_df.head())

,matchid,roundnumber,mapname,objectivelocation,attackers,defenders,attack_win,ATTACK_ASH,ATTACK_BLACKBEARD,ATTACK_BLITZ,...,SITE_MEETING_ROOM,SITE_MEETING_ROOM-EXECUTIVE_OFFICE,SITE_OFFICES,SITE_OPEN_AREA,SITE_SERVER_ROOM-CONTROL_ROOM,SITE_STAFF_SECTION-EXECUTIVE_BEDROOM,SITE_STRIP_CLUB,SITE_TELLERS,SITE_VAULT,SITE_WORKSHOP
0,1522380841,1,CLUB_HOUSE,STRIP_CLUB,"[TWITCH, FUZE, ASH, GLAZ, HIBANA]","[CASTLE, FROST, MIRA, VALKYRIE, MUTE]",0,1,0,0,...,0,0,0,0,0,0,1,0,0,0
1,1522488121,1,PLANE,MEETING_ROOM-EXECUTIVE_OFFICE,"[THERMITE, JACKAL, GLAZ, SLEDGE, MONTAGNE]","[PULSE, FROST, JAGER, VALKYRIE, MIRA]",0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
2,1522488121,2,PLANE,CARGO_HOLD-LUGGAGE_HOLD,"[JACKAL, FUZE, BUCK, ASH, GLAZ]","[PULSE, BANDIT, VALKYRIE, JAGER, MIRA]",1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1522488121,3,PLANE,STAFF_SECTION-EXECUTIVE_BEDROOM,"[GLAZ, TWITCH, ASH, THERMITE, IQ]","[MIRA, VALKYRIE, PULSE, JAGER, BANDIT]",0,1,0,0,...,0,0,0,0,0,1,0,0,0,0
4,1522488121,4,PLANE,CARGO_HOLD-LUGGAGE_HOLD,"[GLAZ, ASH, RECRUIT, IQ, JACKAL]","[DOC, ECHO, JAGER, BANDIT, MIRA]",1,1,0,0,...,0,0,0,0,0,0,0,0,0,0


In [29]:
# Columns that the model should NOT use as input
columns_to_remove = [
    "matchid",
    "roundnumber",
    "mapname",
    "objectivelocation",
    "attackers",
    "defenders",
    "attack_win"
]

# X = all of our ML input features
X = features_df.drop(columns=columns_to_remove)

# y = what we're trying to predict
y = features_df["attack_win"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (65, 70)
y shape: (65,)


In [30]:
print(X.columns.tolist())

['ATTACK_ASH', 'ATTACK_BLACKBEARD', 'ATTACK_BLITZ', 'ATTACK_BUCK', 'ATTACK_CAPITAO', 'ATTACK_FUZE', 'ATTACK_GLAZ', 'ATTACK_HIBANA', 'ATTACK_IQ', 'ATTACK_JACKAL', 'ATTACK_MONTAGNE', 'ATTACK_RECRUIT', 'ATTACK_SLEDGE', 'ATTACK_THATCHER', 'ATTACK_THERMITE', 'ATTACK_TWITCH', 'DEFENSE_BANDIT', 'DEFENSE_CASTLE', 'DEFENSE_CAVEIRA', 'DEFENSE_DOC', 'DEFENSE_ECHO', 'DEFENSE_FROST', 'DEFENSE_JAGER', 'DEFENSE_KAPKAN', 'DEFENSE_MIRA', 'DEFENSE_MUTE', 'DEFENSE_PULSE', 'DEFENSE_RECRUIT', 'DEFENSE_ROOK', 'DEFENSE_SMOKE', 'DEFENSE_TACHANKA', 'DEFENSE_VALKYRIE', 'MAP_BANK', 'MAP_BORDER', 'MAP_CLUB_HOUSE', 'MAP_CONSULATE', 'MAP_KANAL', 'MAP_OREGON', 'MAP_PLANE', 'MAP_SKYSCRAPER', 'MAP_YACHT', 'SITE_1F_BBQ', 'SITE_1F_BEDROOM', 'SITE_2F_TEA_ROOM', 'SITE_2F_WORK_OFFICE', 'SITE_ARCHIVES', 'SITE_ARMORY_LOCKERS', 'SITE_ARSENAL_ROOM', 'SITE_CARGO_HOLD-LUGGAGE_HOLD', 'SITE_CASINO', 'SITE_COCKPIT', 'SITE_ENGINE', 'SITE_GARAGE', 'SITE_KIDS_DORMS-DORMS_MAIN_HALL', 'SITE_KITCHEN-ENGINE_CONTROL', 'SITE_LAUDRY_ROOM-SUP